In [315]:
import numpy as np
import pandas as pd
import seaborn as sns

In [316]:
def split_train_test(x, y, train_size=0.8, random_state=42):
    train_set_size = int(len(x) * train_size)
    
    np.random.seed(random_state)
    shuffled_indices = np.random.permutation(len(x))
        
    train_indices = shuffled_indices[:train_set_size]
    test_indices = shuffled_indices[train_set_size:]
    
    return x.iloc[train_indices], x.iloc[test_indices], y.iloc[train_indices], y.iloc[test_indices]

In [317]:
titanic_data = pd.read_csv("./input/titanic_survival.csv")
titanic_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [318]:
titanic_data.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [319]:
titanic_data = titanic_data.drop(columns=["Name", "PassengerId", "Ticket", "Cabin"])
titanic_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [320]:
age_median = titanic_data["Age"].median()
titanic_data["Age"] = titanic_data["Age"].fillna(age_median)
titanic_data.describe()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.361582,0.523008,0.381594,32.204208
std,0.486592,0.836071,13.019697,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,22.000000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,35.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [321]:
embarked_mode = titanic_data["Embarked"].mode()[0]
titanic_data["Embarked"] = titanic_data["Embarked"].fillna(embarked_mode)
titanic_data.isna().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [322]:
x_train, x_test, y_train, y_test = split_train_test(titanic_data.drop("Survived", axis=1), titanic_data["Survived"])
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((712, 7), (179, 7), (712,), (179,))

In [323]:
x_survived = x_train[y_train == 1]
x_died = x_train[y_train == 0]

survived_prior = len(x_survived) / len(x_train)
died_prior = len(x_died) / len(x_train)

priors = {
    1: survived_prior,
    0: died_prior
}

priors

{1: 0.3890449438202247, 0: 0.6109550561797753}

In [324]:
x_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
709,3,male,28.0,1,1,15.2458,C
439,2,male,31.0,0,0,10.5000,S
840,3,male,20.0,0,0,7.9250,S
720,2,female,6.0,0,1,33.0000,S
39,3,female,14.0,1,0,11.2417,C


In [325]:
gaussian_columns = ["Fare", "Age"]
x_survived_gaussian = x_survived[gaussian_columns]
x_died_gaussian = x_survived[gaussian_columns]

x_survived_means = np.mean(x_survived_gaussian, axis=0)
x_survived_vars = np.var(x_survived_gaussian, axis=0)

x_died_means = np.mean(x_died_gaussian, axis=0)
x_died_vars = np.var(x_died_gaussian, axis=0)

means = {
    1: x_survived_means,
    0: x_died_means
}

variances = {
    1: x_survived_vars,
    0: x_died_vars
}

In [326]:
def compute_gaussian_probability(x, feature_name):
    survived_feature_mean = means[1][feature_name]
    died_feature_mean = means[0][feature_name]
    
    survived_feature_var = variances[1][feature_name]
    died_feature_var = variances[0][feature_name]
    
    a1 = 1 / np.sqrt(2 * np.pi * survived_feature_var)
    b1 = (x[feature_name] - survived_feature_mean) ** 2
    c1 = 2 * survived_feature_var
    
    survived_feature_prob = a1 * np.exp(-b1 / c1)
    
    a2 = 1 / np.sqrt(2 * np.pi * died_feature_var)
    b2 = (x[feature_name] - died_feature_mean) ** 2
    c2 = 2 * died_feature_var
    
    died_feature_prob = a2 * np.exp(-b2 / c2)
    
    return survived_feature_prob, died_feature_prob

In [327]:
categorical_columns = ["Pclass", "Sex", "Embarked", "SibSp", "Parch"]

x_survived_categorical = x_survived[categorical_columns]
x_died_categorical = x_died[categorical_columns]

x_survived_category_counts = {
    "Sex": x_survived_categorical["Sex"].value_counts(),
    "Pclass": x_survived_categorical["Pclass"].value_counts(),
    "Embarked": x_survived_categorical["Embarked"].value_counts(),
    "SibSp": x_survived_categorical["SibSp"].value_counts(),
    "Parch": x_survived_categorical["Parch"].value_counts()
}

x_died_category_counts = {
    "Sex": x_died_categorical["Sex"].value_counts(),
    "Pclass": x_died_categorical["Pclass"].value_counts(),
    "Embarked": x_died_categorical["Embarked"].value_counts(),
    "SibSp": x_died_categorical["SibSp"].value_counts(),
    "Parch": x_died_categorical["Parch"].value_counts()
}

In [328]:
def compute_categorical_probability(x, feature_name, smoothing=1.0):
    vocab_size = len(x_survived_category_counts[feature_name])
    
    survived_category_frequency = x_survived_category_counts[feature_name].get(x[feature_name], 0)
    died_category_frequency = x_died_category_counts[feature_name].get(x[feature_name], 0)
    
    survived_total_frequency = x_survived_category_counts[feature_name].sum()
    died_total_frequency = x_died_category_counts[feature_name].sum()
    
    a1 = survived_category_frequency + smoothing
    b1 = survived_total_frequency + (smoothing * vocab_size) 
    
    survived_feature_prob = a1 / b1
    
    a2 = died_category_frequency + smoothing
    b2 = died_total_frequency + (smoothing * vocab_size)
    
    died_feature_prob = a2 / b2
    
    return survived_feature_prob, died_feature_prob

In [329]:
def compute_class_probabilities(x):
    survived_score = np.log(priors[1])
    died_score = np.log(priors[0])
    
    for feature_name in x.index:
        if feature_name in categorical_columns:
            survived_feature_prob, died_feature_prob = compute_categorical_probability(x, feature_name)
        else:    
            survived_feature_prob, died_feature_prob = compute_gaussian_probability(x, feature_name)
        
        survived_score += np.log(survived_feature_prob)
        died_score += np.log(died_feature_prob)
    
    return survived_score, died_score

In [330]:
def predict(x):
    survived_score, died_score = compute_class_probabilities(x)
    return survived_score > died_score

In [331]:
def test_accuracy(x_test, y_test):
    correct_predictions = 0
    
    for idx, x in x_test.iterrows():
        prediction = predict(x)
        
        if prediction == y_test.loc[idx]:
            correct_predictions += 1
            
    return correct_predictions / len(x_test)

In [ ]:
accuracy = test_accuracy(x_test, y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 78.77
